# Proyecto LOST — MountainCarContinuous-v0 con Q-Learning y Dyna-Q

Notebook integral del Proyecto LOST. Contenido:

1. Setup del ambiente y verificación de los espacios.
2. Demo del `Discretizer`.
3. Smoke test del `QLearningAgent` (entrenamiento corto + curva).
4. Test de persistencia (`save` / `load`).
5. Resumen del **grid search** (12 corridas, resultados precomputados).
6. **Visualización de la política aprendida** (V(s) + π(s)).
7. **Dyna-Q vs Q-Learning** (con y sin shaping).
8. Comparación final de los dos modelos entregables.

El razonamiento, las justificaciones y todos los hallazgos están en `../Documentacion.md`. Los scripts standalone (`grid_search.py`, `train_best.py`, `compare_dyna_q.py`, `visualize_policy.py`) producen los artefactos que esta notebook lee.

In [ ]:
import json
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from discretization import Discretizer
from q_learning_agent import QLearningAgent
from dyna_q_agent import DynaQAgent

## 1. Setup del ambiente

Usamos `render_mode=None` durante entrenamiento (mucho más rápido). Solo activar render para demo visual al final.

In [ ]:
env_id = 'MountainCarContinuous-v0'
env = gym.make(env_id)

print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)
print('Reward terminal (meta):  +100')
print('Reward por step:         -0.1*a^2')
print('Timeout (TimeLimit):     999 steps')

## 2. Discretización

Convertimos `(x, v)` continuos a un par de índices `(xi, vi)` y la fuerza continua `a ∈ [-1, 1]` a uno de `n_actions` valores discretos. La clase `Discretizer` encapsula esta conversión para que el resto del código sea agnóstico de la resolución.

In [ ]:
disc = Discretizer(n_bins_x=40, n_bins_v=40, n_actions=5)
print('q_shape:', disc.q_shape)
print('acciones disponibles:', disc.actions)

obs = np.array([-0.4, 0.02])
print(f'obs={obs} -> state={disc.get_state(obs)}')

## 3. Smoke test del Q-Learning agent

Config media (40×40 bins, 5 acciones) + reward shaping **potential-based** (Ng-Harada-Russell 1999): `F = γ·Φ(s') − Φ(s)` con `Φ = 300·|v|`. Vale la pena destacar que el shaping potential-based es la única forma de hacer reward shaping que **no cambia la política óptima** — un shaping aditivo simple como `r += c·|v|` hace que el agente colapse en una política de *oscilar para acumular bonus* sin llegar a la meta.

500 episodios para validar la pipeline.

In [ ]:
agent = QLearningAgent(
    discretizer=disc,
    alpha=0.1, gamma=0.99,
    epsilon_start=1.0, epsilon_min=0.05, epsilon_decay=0.995,
    reward_shaping=True, shaping_coef=300.0,
    seed=42,
)
history = agent.train_agent(env, episodes=500, verbose_every=100, env_seed=42)

In [ ]:
def plot_history(history, title):
    rewards = np.array(history['rewards'])
    successes = np.array(history['success'])
    window = 50
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    success_rate = np.convolve(successes, np.ones(window)/window, mode='valid')

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    axes[0].plot(rewards, alpha=0.3, label='reward/ep')
    axes[0].plot(np.arange(len(moving_avg)) + window - 1, moving_avg, color='C1', label=f'media movil ({window})')
    axes[0].set_ylabel('Reward acumulado'); axes[0].grid(alpha=0.3); axes[0].legend(); axes[0].set_title(title)
    axes[1].plot(np.arange(len(success_rate)) + window - 1, success_rate, color='C2')
    axes[1].set_ylabel(f'Tasa exito (ventana {window})'); axes[1].set_xlabel('Episodio'); axes[1].set_ylim(-0.05, 1.05); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    return fig

plot_history(history, 'Smoke test - Q-Learning + potential-based shaping')
plt.show()

metrics = agent.test_agent(env, episodes=10)
print(f"\nTest greedy: success_rate={metrics['success_rate']:.0%}, avg_reward={metrics['avg_reward']:.2f}, avg_steps={metrics['avg_steps']:.1f}")

## 4. Persistencia

El `save` guarda Q + config del discretizer + hiperparámetros en un `.pkl`. Permite reconstruir el agente sin conocer la configuración a priori. **Obligatorio para la entrega**.

In [ ]:
agent.save('models/smoke_test.pkl')

loaded = QLearningAgent.load('models/smoke_test.pkl')
print('Modelo cargado, prueba con 5 episodios:', loaded.test_agent(env, episodes=5))

## 5. Grid search — resumen de resultados

El grid search completo se ejecuta con `python3 grid_search.py` (12 corridas, ~30 s). Acá leemos los resultados precomputados desde `grid_search_results.json`. Estrategia: **one-at-a-time** sobre una config base.

Las corridas exitosas (test_success=100%) se ordenan por menor `test_avg_steps` (mejor política).

In [ ]:
results = json.load(open('grid_search_results.json'))
print(f"{'Run':<22s} {'conv':>5s} {'succ':>6s} {'reward':>8s} {'steps':>7s}")
print('-'*55)
for r in results:
    conv = str(r['convergence_ep_50w_0.9'])
    print(f"{r['name']:<22s} {conv:>5s} {r['test_success_rate']:>6.0%} {r['test_avg_reward']:>8.2f} {r['test_avg_steps']:>7.1f}")

Curva de aprendizaje de las 12 corridas y resumen de métricas (generados por `grid_search.py`):

In [ ]:
from IPython.display import Image, display
display(Image('plots/grid_search_curves.png'))
display(Image('plots/grid_search_summary.png'))

**Mejor config:** `alpha_0.05` (bins=40, n_actions=5, α=0.05, γ=0.99, ε_decay=0.995, shaping coef=300).

Entrenamos este modelo con 2000 episodios (script `train_best.py`) y lo guardamos como `models/q_learning_best.pkl` — éste es el entregable de Q-Learning.

In [ ]:
best_ql = QLearningAgent.load('models/q_learning_best.pkl')
best_ql_metrics = best_ql.test_agent(env, episodes=50)
print('Q-Learning best, 50 episodios test greedy:')
for k, v in best_ql_metrics.items():
    if k != 'rewards':
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

display(Image('plots/q_learning_best_curve.png'))

## 6. Visualización de la política aprendida

Para verificar que la política tiene sentido (más allá de las métricas escalares), graficamos V(s) y π(s) en el espacio de estado (x, v), enmascarando estados no visitados durante el entrenamiento.

**Estrategia esperada (pump-and-go):** cuando v > 0, empujar +1; cuando v < 0, empujar −1. Acumula momento hasta poder subir.

In [ ]:
display(Image('plots/q_learning_best_policy.png'))

## 7. Dyna-Q vs Q-Learning

Implementamos `DynaQAgent` (Sutton & Barto §8.2) heredando de `QLearningAgent`. La idea: por cada step real, hacemos `n` updates de Q usando el modelo aprendido.

Hicimos dos experimentos:

- **Con shaping** (mismo setup que el mejor Q-Learning): `compare_dyna_q.py`.
- **Sin shaping** (escenario sparso donde S&B predice ventaja del planning).

### 7.1 Con shaping

In [ ]:
dq_results = json.load(open('dyna_q_comparison.json'))
print(f"{'n':>3s} {'conv':>5s} {'time':>7s} {'succ':>6s} {'reward':>8s} {'steps':>7s} {'|model|':>8s}")
print('-'*55)
for r in dq_results:
    conv = str(r['convergence_ep'])
    print(f"{r['planning_steps']:>3d} {conv:>5s} {r['train_time_s']:>6.1f}s {r['test_success_rate']:>6.0%} {r['test_avg_reward']:>8.2f} {r['test_avg_steps']:>7.1f} {r['model_size']:>8d}")

In [ ]:
display(Image('plots/dyna_q_comparison.png'))
display(Image('plots/dyna_q_convergence_vs_time.png'))

**Hallazgo (con shaping):** Q-Learning puro converge más rápido que cualquier Dyna-Q. El planning, con shaping ya tan informativo, agrega ruido al inicio en lugar de aprendizaje útil. El tiempo de cómputo crece lineal en n, como predice S&B.

### 7.2 Sin shaping (escenario sparso, bins=20)

Para demostrar que Dyna-Q SÍ ayuda en su régimen natural (reward muy sparso), repetimos sin shaping con la config gruesa (bins=20, n_actions=3) — con bins=40 ni siquiera n=50 logra aprender en 800 ep, el espacio queda demasiado sparso.

In [ ]:
dq_ns = json.load(open('dyna_q_no_shaping.json'))
print(f"{'n':>3s} {'conv':>5s} {'train_succ':>10s} {'test_succ':>10s} {'test_steps':>10s}")
print('-'*50)
for r in dq_ns:
    conv = str(r['conv'])
    print(f"{r['n']:>3d} {conv:>5s} {r['succ_train_last100']:>10.0%} {r['test_succ']:>10.0%} {r['test_steps']:>10.1f}")

display(Image('plots/dyna_q_no_shaping.png'))

**Hallazgo (sin shaping):** Q-Learning puro nunca aprende. Solo Dyna-Q con n=50 logra una política aceptable (75% éxito en test). El planning es la diferencia entre aprender y no aprender en este régimen.

**Conclusión general:** la utilidad de Dyna-Q depende del régimen — cuando cada step real ya lleva una señal informativa (gracias al shaping), Q-Learning solo es suficiente. Cuando el reward es sparso, el planning compensa la escasez de experiencia.

## 8. Comparación final de modelos entregables

In [ ]:
best_dq = DynaQAgent.load('models/dyna_q_best.pkl')
best_dq_metrics = best_dq.test_agent(env, episodes=50)

print('Comparacion final (50 episodios test greedy):')
print('  q_learning_best (n=0,  alpha=0.05, bins=40, 2000 ep):')
print(f"    test_succ={best_ql_metrics['success_rate']:.0%}, reward={best_ql_metrics['avg_reward']:.2f}, steps={best_ql_metrics['avg_steps']:.1f}")
print('  dyna_q_best (n=5,  alpha=0.05, bins=40,  400 ep):')
print(f"    test_succ={best_dq_metrics['success_rate']:.0%}, reward={best_dq_metrics['avg_reward']:.2f}, steps={best_dq_metrics['avg_steps']:.1f}")

In [ ]:
env.close()